In [ ]:
# Cell 1 — Load config and utilities
%run /home/jovyan/work/setup/config.py
import sys; sys.path.insert(0, "/home/jovyan/work")
from utils.dq import dq_check, write_dq_log
from utils.delta_utils import save_layer

In [ ]:
# Cell 2 — Read Silver + all dimensions
from pyspark.sql.functions import monotonically_increasing_id, col

df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/silver_beverage_sales_enriched")
dim_date  = spark.read.format("delta").load(f"{GOLD_PATH}/dim_date")
dim_prod  = spark.read.format("delta").load(f"{GOLD_PATH}/dim_product")
dim_chan  = spark.read.format("delta").load(f"{GOLD_PATH}/dim_channel")
dim_geo   = spark.read.format("delta").load(f"{GOLD_PATH}/dim_geography")

silver_count = df_silver.count()
print(f"Silver rows to process: {silver_count}")

In [ ]:
# Cell 3 — Resolve surrogate keys via joins
df_fact = (
    df_silver
    .join(dim_date.select("date_sk", "full_date"),
          on="full_date", how="left")
    .join(dim_prod.select("product_sk", "ce_brand_flvr", "pkg_cat", "tsr_pckg_nm"),
          on=["ce_brand_flvr", "pkg_cat", "tsr_pckg_nm"], how="left")
    .join(dim_chan.select("channel_sk", "trade_chnl_desc"),
          on="trade_chnl_desc", how="left")
    .join(dim_geo.select("geography_sk", "region"),
          on="region", how="left")
    .select("date_sk", "product_sk", "channel_sk", "geography_sk",
            col("dollar_volume"), col("_source_file"), col("_ingest_ts"))
    .withColumn("sale_sk", monotonically_increasing_id())
    .select("sale_sk", "date_sk", "product_sk", "channel_sk", "geography_sk",
            "dollar_volume", "_source_file", "_ingest_ts")
)

fact_count = df_fact.count()
print(f"Fact rows: {fact_count} | Silver was: {silver_count}")
assert fact_count == silver_count, "Row count mismatch between Silver and Fact!"

In [ ]:
# Cell 4 — Write fact_sales to Delta + PostgreSQL
save_layer(df_fact, "fact_sales", GOLD_PATH, PG_WRITE_PROPS)

In [ ]:
# Cell 5 — DQ Gold checks
import uuid
run_id       = str(uuid.uuid4())
null_date_sk = df_fact.filter(col("date_sk").isNull()).count()
null_prod_sk = df_fact.filter(col("product_sk").isNull()).count()
null_chan_sk = df_fact.filter(col("channel_sk").isNull()).count()
null_geo_sk  = df_fact.filter(col("geography_sk").isNull()).count()

checks = [
    dq_check(run_id, "gold", "fact_sales", "row_count_eq_silver",   str(silver_count), str(fact_count)),
    dq_check(run_id, "gold", "fact_sales", "no_null_date_sk",       "0", str(null_date_sk)),
    dq_check(run_id, "gold", "fact_sales", "no_null_product_sk",    "0", str(null_prod_sk)),
    dq_check(run_id, "gold", "fact_sales", "no_null_channel_sk",    "0", str(null_chan_sk)),
    dq_check(run_id, "gold", "fact_sales", "no_null_geography_sk",  "0", str(null_geo_sk)),
]
write_dq_log(spark, checks, GOLD_PATH)